# Imports

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset

from transformers import (
    CLIPModel,
    CLIPProcessor,
    TrainingArguments,
    Trainer
)

# Config

In [ ]:
import json
from pathlib import Path
CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

PROCESSED_DIR = Path(config["processed_dir"])
CHECKPOINT_DIR = Path(config["checkpoint_dir"])

MODEL_NAME = "laion/CLIP-ViT-B-16-laion2B-s34B-b88K"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)

print("Copying archive to local disk...")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

LOCAL_ARCHIVE_COPY.unlink()


print("This is LOCAL Colab disk -- it resets when the runtime disconnects.")
print("Re-run this cell at the start of every session before training.")


Copying archive to local disk...
Copy done in 905.7s
Extracting locally...
Extract done in 631.7s

100,918 files extracted to /content/CholecT50
This is LOCAL Colab disk -- it resets when the runtime disconnects.
Re-run this cell at the start of every session before training.


# Load Datasets

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "matched_train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "matched_val.parquet")

# Filter out rows where 'image_full_path' is None or NaN
train_df = train_df.dropna(subset=['image_full_path'])
val_df = val_df.dropna(subset=['image_full_path'])

print(train_df.shape)
print(val_df.shape)

train_small = train_df
val_small = val_df

print("\nTask distribution (train):")
print(train_small["task"].value_counts())


(91000, 18)
(9000, 18)

Task distribution (train):
task
Action Recognition              13000
Instrument Recognition          13000
Phase Recognition               13000
Safety Assessment               13000
Surgical Image Captioning       13000
Tissue and Organ Recognition    13000
Triplet Recognition             13000
Name: count, dtype: int64


# Load Processor and Model

In [ ]:
new_model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(new_model_name)

model = CLIPModel.from_pretrained(new_model_name)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

# Dataset

In [ ]:
class SurgSigmaDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row.image_full_path).convert("RGB")

        return {
            "image": image,
            "text": row.text
        }

# Processor

In [ ]:
def collate_fn(batch):

    images = [x["image"] for x in batch]
    texts = [x["text"] for x in batch]

    return processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

# Custom Trainer

In [ ]:
import torch

class CLIPTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        outputs = model(
            **inputs,
            return_loss=True
        )

        loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

    # Add this overridden prediction_step method
    def prediction_step(
        self, model, inputs, prediction_loss_only, ignore_keys=None
    ):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            outputs = model(**inputs, return_loss=True)
            # The CLIPModel returns a CLIPOutput object which has a .loss attribute
            loss = outputs.loss
            # For CLIP, predictions are typically embeddings (image and text)
            # If we wanted to compute metrics beyond loss, you'd extract them here
            # For now, we just ensure loss is returned.
            # The Trainer expects (loss, predictions, labels)
            # Since CLIP doesn't have standard 'labels', we can pass None or a dummy.
            return (loss, None, None)

# Training Args

In [ ]:
training_args = TrainingArguments(

    output_dir=CHECKPOINT_DIR,

    dataloader_num_workers= 8,

    dataloader_persistent_workers=True,

    learning_rate=5e-6,

    weight_decay=0.01,

    per_device_train_batch_size=256,

    per_device_eval_batch_size=256,

    gradient_accumulation_steps = 1,

    num_train_epochs=5,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    save_total_limit=2,

    bf16 = True,

    report_to="none",

    remove_unused_columns=False
)

# Trainer

In [ ]:
def compute_metrics_for_clip(eval_pred):
    # This function is provided to ensure the Trainer correctly registers metrics.
    # For a loss-only scenario like CLIP, returning an empty dictionary is often sufficient
    # for the Trainer to then automatically include 'eval_loss'.
    return {"dummy_eval_metric": 0.0}

trainer = CLIPTrainer(

    model=model,

    args=training_args,

    train_dataset=SurgSigmaDataset(train_small),

    eval_dataset=SurgSigmaDataset(val_small),

    data_collator=collate_fn,

    compute_metrics = compute_metrics_for_clip

)

# Train

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.068049,4.739971
2,3.761752,4.782388
3,3.551801,4.884846
4,3.342874,4.956720
5,3.175621,5.113046


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1780, training_loss=3.6620337325535464, metrics={'train_runtime': 1411.6675, 'train_samples_per_second': 322.314, 'train_steps_per_second': 1.261, 'total_flos': 2.646598407453e+16, 'train_loss': 3.6620337325535464, 'epoch': 5.0})

# Save Model

In [ ]:
trainer.save_model(CHECKPOINT_DIR / "best_openclip")

processor.save_pretrained(CHECKPOINT_DIR / "best_openclip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/Surgical-VLM/checkpoints/best_openclip/processor_config.json']